In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


I0000 00:00:1785391814.371539  125520 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785391814.511426  125520 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785391817.226015  125520 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
IMG_SIZE = 128
BATCH_SIZE = 16
NUM_CLASSES = 7

In [3]:
#Load Dataset

train_df = pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/train.csv")
valid_df = pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/val.csv")
test_df = pd.read_csv("//home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)
metadata = pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/HAM10000_metadata.csv")

metadata.head()

(7010, 8)
(1502, 8)
(1503, 8)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [4]:
image_dir1 = "/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/HAM10000_images_part_1"
image_dir2 = "/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/HAM10000_images_part_2"

image_path = {}

for folder in [image_dir1, image_dir2]:

    for file in os.listdir(folder):

        image_id = file.split(".")[0]

        image_path[image_id] = os.path.join(folder, file)

metadata["path"] = metadata["image_id"].map(image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...


In [5]:
encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2


In [6]:
train_df, temp_df = train_test_split(

    metadata,

    test_size=0.30,

    stratify=metadata["label"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 9)
(1502, 9)
(1503, 9)


In [7]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [8]:
train_df["path"] = train_df["path"].astype(str)
val_df["path"] = val_df["path"].astype(str)
test_df["path"] = test_df["path"].astype(str)

In [9]:
classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [10]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.2,

    horizontal_flip=True,

    vertical_flip=True
)

test_datagen = ImageDataGenerator(

    rescale=1./255
)

In [11]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [12]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    horizontal_flip=True,

    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]

)

test_datagen = ImageDataGenerator(

    rescale=1./255

)

In [13]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True

)

Found 7010 validated image filenames belonging to 7 classes.


# EarlyStop

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Dense
)
from tensorflow.keras.optimizers import Adam

cnn_bn = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(256,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(512,activation="relu"),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_bn.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 331,991 (1.27 MB)

 Trainable params: 110,663 (432.28 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 221,328 (864.57 KB)

In [21]:
cnn_bn.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [22]:
early_stop = EarlyStopping(

    monitor="val_accuracy",

    mode="max",

    patience=3,

    restore_best_weights=True,

    verbose=1

)

In [23]:
history_bn_es = cnn_bn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=20,

    class_weight=class_weights,

    callbacks=[early_stop]

)

Epoch 1/20


I0000 00:00:1785338760.977839  114054 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


439/439 ━━━━━━━━━━━━━━━━━━━━ 178s 402ms/step - accuracy: 0.4223 - loss: 1.8270 - val_accuracy: 0.4820 - val_loss: 1.4361
Epoch 2/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 183s 417ms/step - accuracy: 0.4787 - loss: 1.6757 - val_accuracy: 0.4780 - val_loss: 1.4371
Epoch 3/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 168s 382ms/step - accuracy: 0.4783 - loss: 1.6270 - val_accuracy: 0.4760 - val_loss: 1.4481
Epoch 4/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 171s 389ms/step - accuracy: 0.4652 - loss: 1.6040 - val_accuracy: 0.4514 - val_loss: 1.4876
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_bn.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_bn.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_bn.evaluate(test_generator, verbose=0)

pred = cnn_bn.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [28]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.4746077060699463 0.4820239543914795 0.4710578918457031 0.6577451875257033 0.47105788423153694 0.525669425590849


In [ ]:
comparison_basic_lr = pd.DataFrame({

    "Metric":[
        "Train Accuracy",
        "Validation Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Phase 4":[
        0.480328081,
        0.522436774,
        0.494650699,
        0.438463913,
        0.424650699,
        0.419771229
    ],

    "After Batch":[
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ]

})

comparison_basic_lr["Improvement"] = (

    comparison_basic_lr["After Batch"] -

    comparison_basic_lr["Phase 4"]

)

comparison_basic_lr

# learning rate

In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Dense
)
from tensorflow.keras.optimizers import Adam

cnn_bn = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(256,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(512,activation="relu"),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_bn.summary()

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 525,511 (2.00 MB)

 Trainable params: 524,551 (2.00 MB)

 Non-trainable params: 960 (3.75 KB)

In [30]:
cnn_bn.compile(

    optimizer=Adam(learning_rate=0.001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [31]:
reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-6,

    verbose=1

)

In [33]:
history_lr = cnn_bn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[reduce_lr]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 323s 735ms/step - accuracy: 0.3422 - loss: 2.0045 - val_accuracy: 0.0340 - val_loss: 2.6937 - learning_rate: 0.0010
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 526s 1s/step - accuracy: 0.4184 - loss: 1.6461 - val_accuracy: 0.3808 - val_loss: 1.5166 - learning_rate: 0.0010
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 508s 1s/step - accuracy: 0.4565 - loss: 1.5371 - val_accuracy: 0.1019 - val_loss: 2.0937 - learning_rate: 0.0010
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4431 - loss: 1.4811
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
439/439 ━━━━━━━━━━━━━━━━━━━━ 526s 1s/step - accuracy: 0.4431 - loss: 1.4811 - val_accuracy: 0.3063 - val_loss: 1.9539 - learning_rate: 0.0010
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 500s 1s/step - accuracy: 0.4920 - loss: 1.3808 - val_accuracy: 0.4900 - val_loss: 1.3010 - learning_rate: 5.0000e-04


In [35]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_bn.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_bn.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_bn.evaluate(test_generator, verbose=0)

pred = cnn_bn.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [37]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.49700427055358887 0.4900133013725281 0.4883566200733185 0.742684603733933 0.48835662009314706 0.554490662450181


In [36]:
cnn_bn.save("models2/3_lr.keras")

# SGD


In [38]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Dense
)
from tensorflow.keras.optimizers import SGD

cnn_bn = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(256,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(512,activation="relu"),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_bn.summary()

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 525,511 (2.00 MB)

 Trainable params: 524,551 (2.00 MB)

 Non-trainable params: 960 (3.75 KB)

In [39]:
cnn_bn.compile(

    optimizer=SGD(

        learning_rate=0.01,

        momentum=0.9

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [40]:
history_sgd = cnn_bn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 326s 739ms/step - accuracy: 0.3061 - loss: 2.0423 - val_accuracy: 0.3375 - val_loss: 1.7482
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 317s 721ms/step - accuracy: 0.3622 - loss: 1.8305 - val_accuracy: 0.4168 - val_loss: 1.5740
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 505s 1s/step - accuracy: 0.3726 - loss: 1.8129 - val_accuracy: 0.6185 - val_loss: 2.0992
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 483s 1s/step - accuracy: 0.3612 - loss: 1.8188 - val_accuracy: 0.2683 - val_loss: 1.8337
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 488s 1s/step - accuracy: 0.4140 - loss: 1.7334 - val_accuracy: 0.4634 - val_loss: 1.3573


In [41]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_bn.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_bn.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_bn.evaluate(test_generator, verbose=0)

pred = cnn_bn.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [42]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.47860199213027954 0.46338215470314026 0.46041250228881836 0.6668804320917388 0.46041250831669994 0.5138974648651952


In [43]:
cnn_bn.save("models2/3_sgd.keras")

# # RMSprop


In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Dense
)
from tensorflow.keras.optimizers import RMSprop

In [45]:
cnn_bn = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(256,(3,3),activation="relu",padding="same"),
    BatchNormalization(),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(512,activation="relu"),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_bn.summary()

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 525,511 (2.00 MB)

 Trainable params: 524,551 (2.00 MB)

 Non-trainable params: 960 (3.75 KB)

In [46]:
cnn_bn.compile(

    optimizer=RMSprop(

        learning_rate=0.001

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [47]:
history_rmsprop = cnn_bn.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 312s 707ms/step - accuracy: 0.3725 - loss: 1.8877 - val_accuracy: 0.1145 - val_loss: 2.2664
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 301s 684ms/step - accuracy: 0.4160 - loss: 1.6126 - val_accuracy: 0.1531 - val_loss: 2.3748
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 591s 1s/step - accuracy: 0.4552 - loss: 1.5634 - val_accuracy: 0.4933 - val_loss: 1.2665
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 315s 717ms/step - accuracy: 0.4695 - loss: 1.5022 - val_accuracy: 0.4061 - val_loss: 1.5013
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 303s 690ms/step - accuracy: 0.4883 - loss: 1.4973 - val_accuracy: 0.2876 - val_loss: 1.7184


In [48]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = eeeeww.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_bn.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_bn.evaluate(test_generator, verbose=0)

pred = cnn_bn.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [49]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.3426533639431 0.2876165211200714 0.27811044454574585 0.7035489286981987 0.27811044577511645 0.3220379261184965


In [53]:
cnn_bn.save("models2/3_rms.keras")

# Batch Wise Comparison

In [54]:
train_generator_32 = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=True

)

val_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

test_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [55]:
cnn_bn.compile(

    optimizer=Adam(learning_rate=0.001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [56]:
history_batch32 = cnn_bn.fit(

    train_generator_32,

    validation_data=val_generator_32,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 497s 2s/step - accuracy: 0.5111 - loss: 1.3254 - val_accuracy: 0.4254 - val_loss: 1.5483
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 387s 2s/step - accuracy: 0.5252 - loss: 1.2561 - val_accuracy: 0.5013 - val_loss: 1.2533
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.5399 - loss: 1.2422 - val_accuracy: 0.4893 - val_loss: 1.4055
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 330s 1s/step - accuracy: 0.5412 - loss: 1.2051 - val_accuracy: 0.5293 - val_loss: 1.2066
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 333s 2s/step - accuracy: 0.5481 - loss: 1.2222 - val_accuracy: 0.4501 - val_loss: 1.3979


In [57]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_bn.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_bn.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_bn.evaluate(test_generator, verbose=0)

pred = cnn_bn.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [58]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.4362339377403259 0.45006656646728516 0.4424484372138977 0.7383236475561298 0.4424484364604125 0.5203414663008972


In [59]:
cnn_bn.save("models2/3_batch.keras")

# Hyperparameter Optimization

In [24]:
from keras_tuner import HyperModel
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.optimizers import SGD

class CNNBNHyperModel(HyperModel):

    def build(self, hp):

        model = Sequential()

        model.add(

            Conv2D(

                filters=hp.Choice(
                    "filters1",
                    [32,64]
                ),

                kernel_size=(3,3),

                activation="relu",

                input_shape=(IMG_SIZE,IMG_SIZE,3)

            )

        )

        model.add(BatchNormalization())

        model.add(MaxPooling2D())

        model.add(

            Conv2D(

                filters=hp.Choice(
                    "filters2",
                    [64,128]
                ),

                kernel_size=(3,3),

                activation="relu"

            )

        )

        model.add(BatchNormalization())

        model.add(MaxPooling2D())

        model.add(Flatten())

        model.add(

            Dense(

                units=hp.Choice(
                    "dense_units",
                    [64,128,256]
                ),

                activation="relu"

            )

        )

        model.add(

            Dense(
                NUM_CLASSES,
                activation="softmax"
            )

        )

        optimizer = hp.Choice(

            "optimizer",

            ["adam","rmsprop"]

        )

        learning_rate = hp.Choice(

            "learning_rate",

            [1e-3,5e-4,1e-4]

        )

        if optimizer=="adam":

            opt = Adam(learning_rate=learning_rate)

        elif optimizer=="rmsprop":

            opt = RMSprop(learning_rate=learning_rate)

        
            

        model.compile(

            optimizer=opt,

            loss="categorical_crossentropy",

            metrics=["accuracy"]

        )

        return model

In [18]:
import keras_tuner as kt

from keras_tuner import HyperModel
from keras_tuner import RandomSearch
tuner = RandomSearch(

    CNNBNHyperModel(),

    objective="val_accuracy",

    max_trials=3,          # instead of 10

    executions_per_trial=1,

    directory="cnn_bn_tuning",

    project_name="cnn_bn"

)

tuner.search(

    train_generator,

    validation_data=val_generator,

    epochs=5

)
callbacks=[
    EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

Trial 3 Complete [00h 41m 42s]
val_accuracy: 0.6804261207580566

Best val_accuracy So Far: 0.6804261207580566
Total elapsed time: 01h 33m 11s


In [19]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print(best_hp.values)

{'filters1': 64, 'filters2': 64, 'dense_units': 128, 'optimizer': 'adam', 'learning_rate': 0.001}


In [20]:
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 343s 776ms/step - accuracy: 0.2720 - loss: 12.3829 - val_accuracy: 0.0213 - val_loss: 5.0947
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 474s 987ms/step - accuracy: 0.2862 - loss: 3.0553 - val_accuracy: 0.2743 - val_loss: 1.8396
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 618s 1s/step - accuracy: 0.1556 - loss: 1.9953 - val_accuracy: 0.1085 - val_loss: 3.2820
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 600s 1s/step - accuracy: 0.2374 - loss: 2.3854 - val_accuracy: 0.3003 - val_loss: 9.2346
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 560s 1s/step - accuracy: 0.2397 - loss: 2.1677 - val_accuracy: 0.3229 - val_loss: 1.6061


In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = best_model.evaluate(train_generator, verbose=0)

val_loss, val_acc = best_model.evaluate(val_generator, verbose=0)

test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)

pred = best_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [22]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.3049928545951843 0.3229027986526489 0.31470391154289246 0.6537641279723714 0.3147039254823686 0.3979121324817096


In [23]:
best_model.save("models2/best_model_cnn_bn.keras")

In [26]:

import pandas as pd
 
batchnorm_phase5 = pd.DataFrame({
 
    "Technique": [

        "Early Stopping",

        "Learning Rate Scheduling",

        "SGD",

        "RMSprop",

        "Batch Size = 32",

        "Hyperparameter Tuning"

    ],
 
    "Train Accuracy": [

        0.4746077060699463,

        0.49700427055358887,

        0.47860199213027954,

        0.3426533639431,

        0.4362339377403259,

        0.3049928545951843

    ],
 
    "Validation Accuracy": [

        0.4820239543914795,

        0.4900133013725281,

        0.46338215470314026,

        0.2876165211200714,

        0.45006656646728516,

        0.3229027986526489

    ],
 
    "Test Accuracy": [

        0.4710578918457031,

        0.4883566200733185,

        0.46041250228881836,

        0.27811044454574585,

        0.4424484372138977,

        0.31470391154289246

    ],
 
    "Precision": [

        0.6577451875257033,

        0.742684603733933,

        0.6668804320917388,

        0.7035489286981987,

        0.7383236475561298,

        0.6537641279723714

    ],
 
    "Recall": [

        0.47105788423153694,

        0.48835662009314706,

        0.46041250831669994,

        0.27811044577511645,

        0.4424484364604125,

        0.3147039254823686

    ],
 
    "F1 Score": [

        0.525669425590849,

        0.554490662450181,

        0.5138974648651952,

        0.3220379261184965,

        0.5203414663008972,

        0.3979121324817096

    ]

})
 
batchnorm_phase5 = batchnorm_phase5.sort_values(

    by="Test Accuracy",

    ascending=False

).reset_index(drop=True)
 
batchnorm_phase5
 

,Technique,Train Accuracy,Validation Accuracy,Test Accuracy,Precision,Recall,F1 Score
0,Learning Rate Scheduling,0.497004,0.490013,0.488357,0.742685,0.488357,0.554491
1,Early Stopping,0.474608,0.482024,0.471058,0.657745,0.471058,0.525669
2,SGD,0.478602,0.463382,0.460413,0.666880,0.460413,0.513897
3,Batch Size = 32,0.436234,0.450067,0.442448,0.738324,0.442448,0.520341
4,Hyperparameter Tuning,0.304993,0.322903,0.314704,0.653764,0.314704,0.397912
5,RMSprop,0.342653,0.287617,0.278110,0.703549,0.278110,0.322038
